# Lab 1: AutoEncoder 이상탐지
## 목표: 정상 패턴만 학습해서 이상 신호를 자동으로 탐지한다

### 핵심 개념
- AutoEncoder: 입력을 압축(인코더) → 복원(디코더)하는 신경망
- 정상 데이터만 학습 → 이상 데이터는 복원이 잘 안 됨 (재구성 오차 높음)
- 재구성 오차(Reconstruction Error) > 임계값 → 이상 판정

### 사용 데이터
- NASA C-MAPSS FD001 (항공 엔진 수명 데이터)
- KAMP 데이터셋 id=26 (설비 센서 시계열)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, roc_auc_score

# 재현성
torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 로드 완료")
print(f"PyTorch 버전: {torch.__version__}")

## 1단계: 데이터 준비
NASA C-MAPSS FD001 데이터셋 사용
- 컬럼: unit_id, cycle, op1, op2, op3, s1~s21 (센서 21개)
- 21개 센서 → 이상탐지에 유용한 센서 선택

In [ ]:
# NASA C-MAPSS 데이터 시뮬레이션 (실제 데이터 없을 때 사용)
# 실제 사용: pd.read_csv('data/nasa_cmapss/FD001.txt', sep=' ', header=None)

def generate_cmapss_simulation(n_units=5, max_cycles=200, n_sensors=14):
    """NASA C-MAPSS FD001 데이터 시뮬레이션"""
    records = []
    for unit in range(1, n_units + 1):
        cycles = np.random.randint(100, max_cycles)
        rul_true = cycles - np.arange(cycles)
        for cycle in range(1, cycles + 1):
            # 정상 구간: 안정적 신호, 고장 구간: 점진적 열화
            degradation = cycle / cycles
            sensors = np.random.normal(loc=1.0 + 0.5*degradation, 
                                        scale=0.05 + 0.1*degradation, 
                                        size=n_sensors)
            records.append([unit, cycle] + list(sensors) + [cycles - cycle])
    
    cols = ['unit_id', 'cycle'] + [f's{i}' for i in range(1, n_sensors+1)] + ['RUL']
    return pd.DataFrame(records, columns=cols)

df = generate_cmapss_simulation()
print(f"데이터 크기: {df.shape}")
print(f"유닛 수: {df['unit_id'].nunique()}")
display(df.head())

In [ ]:
# 정상/이상 라벨링: RUL > 50 → 정상, RUL <= 50 → 이상
df['label'] = (df['RUL'] <= 50).astype(int)

# 특성 선택 (센서 14개)
feature_cols = [c for c in df.columns if c.startswith('s')]

# 정규화
scaler = MinMaxScaler()
X = scaler.fit_transform(df[feature_cols])

# 정상 데이터만으로 학습셋 구성
X_normal = X[df['label'] == 0]
X_test = X
y_test = df['label'].values

print(f"학습 데이터(정상): {X_normal.shape}")
print(f"테스트 데이터: {X_test.shape}")
print(f"이상 비율: {y_test.mean():.1%}")

## 2단계: AutoEncoder 모델 정의

In [ ]:
class AutoEncoder(nn.Module):
    """제조 센서 데이터용 AutoEncoder"""
    def __init__(self, input_dim, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = AutoEncoder(input_dim=len(feature_cols), latent_dim=4)
print(model)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

X_train_t = torch.FloatTensor(X_normal).to(device)

losses = []
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, X_train_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/100 | Loss: {loss.item():.6f}")

plt.figure(figsize=(10,3))
plt.plot(losses)
plt.title('AutoEncoder 학습 손실 (Loss)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
X_test_t = torch.FloatTensor(X_test).to(device)
with torch.no_grad():
    recon = model(X_test_t).cpu().numpy()

# 재구성 오차 계산
recon_error = np.mean((X_test - recon)**2, axis=1)

# 임계값: 정상 데이터 재구성 오차의 95 퍼센타일
threshold = np.percentile(recon_error[y_test==0], 95)
y_pred = (recon_error > threshold).astype(int)

print(f"임계값(Threshold): {threshold:.6f}")
print("\n=== 이상탐지 성능 ===")
print(classification_report(y_test, y_pred, target_names=['정상','이상']))
print(f"AUC-ROC: {roc_auc_score(y_test, recon_error):.4f}")

# 시각화
plt.figure(figsize=(12, 4))
plt.plot(recon_error, alpha=0.5, label='재구성 오차')
plt.axhline(threshold, color='r', linestyle='--', label=f'임계값 ({threshold:.4f})')
plt.scatter(np.where(y_test==1)[0], recon_error[y_test==1], 
            color='red', s=10, label='실제 이상', zorder=5)
plt.title('AutoEncoder 이상탐지 결과')
plt.legend()
plt.tight_layout()
plt.show()